<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Medidores Eléctricos en un Clúster que se Cae ⚡
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 01
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/homeworks/01_Procesamiento_Almacenamiento_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 01: Procesamiento y Almacenamiento Distribuido** ([README del módulo](../01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/README.md)) con un problema **nuevo**: las lecturas de **medidores eléctricos inteligentes** de una cooperativa **ficticia** que debe guardarse y procesarse en un clúster. Todo se implementa en **Python puro** (sin Spark), como en el módulo.

Al terminar serás capaz de:

1. **Razonar con réplicas y quórums** ($R + W > N$), y ver con una partición de red qué sacrifican las políticas **CP** y **AP** del teorema CAP.
2. Escribir un **MapReduce** completo (mapa, *combiner*, partición por *hash*, *shuffle*, reducción), medir lo que ahorra el *combiner* y **arreglar el sesgo de claves** con sal.
3. Simular un **mini-HDFS**: bloques, réplicas con conciencia de *racks*, caída de un nodo o de un *rack* y re-replicación, y el costo de los **archivos pequeños**.
4. Construir un **mini-HBase**: filas ordenadas, versiones por *timestamp*, escaneos por rango y prefijo, y evitar un **punto caliente** con claves con sal.

> 📚 **Repasa si lo necesitas:** [Opciones de procesamiento (CAP)](../01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/00_Opciones_de_Procesamiento_Hadoop_Spark_NoSQL_SQL.ipynb) · [MapReduce y sus limitaciones](../01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/01_MapReduce_y_sus_Limitaciones.ipynb) · [Frameworks distribuidos (HDFS, Spark, HBase)](../01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/02_Frameworks_Distribuidos_HDFS_Spark_HBase.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet, sin Spark y sin archivos externos** (datos sintéticos con semilla fija; los medidores y municipios son **inventados**). Bibliotecas: `numpy`, `pandas` y la biblioteca estándar (`zlib`, `bisect`, `itertools`, `collections`). Cada celda corre en menos de un segundo.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import bisect
import itertools
import zlib
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

SEMILLA = 21
MUNICIPIOS = ["Tunja", "Duitama", "Sogamoso", "Chiquinquirá", "Paipa", "Villa de Leyva",
              "Samacá", "Ráquira", "Moniquirá", "Garagoa", "Soatá", "Ventaquemada"]


def generar_lecturas(semilla=SEMILLA, n=24_000):
    """Lecturas (municipio, kWh). La capital concentra el 55 % de los medidores (sesgo de claves)."""
    rng = np.random.default_rng(semilla)
    p = np.array([0.55] + [0.45 / 11] * 11)
    mun = rng.choice(MUNICIPIOS, n, p=p)
    kwh = np.round(rng.gamma(4.0, 30.0, n), 2)
    return [(str(m), float(k)) for m, k in zip(mun, kwh)]


def dividir(lista, n):
    """Parte `lista` en n trozos contiguos cuyos tamaños difieren a lo sumo en 1."""
    k, r = divmod(len(lista), n)
    trozos, i = [], 0
    for j in range(n):
        m = k + (j < r)
        trozos.append(lista[i:i + m])
        i += m
    return trozos


print("Municipios:", len(MUNICIPIOS), "| lecturas de ejemplo:", generar_lecturas(n=3))

---
## Reto 1: Réplicas, quórums y una red partida 🕸️

**Contexto:** el saldo de un cliente de la cooperativa se guarda en $N = 5$ réplicas. Cada réplica es un diccionario `clave -> (versión, valor)`. Una escritura necesita el acuse de $W$ réplicas y una lectura consulta a $R$ réplicas. Un corte de red separa las réplicas `LADO_A = [0, 1, 2]` y `LADO_B = [3, 4]`.

**Enunciado**
1. `escribir(cluster, clave, valor, version, W, alcanzables)`: si hay menos de `W` réplicas alcanzables, **rechaza** (devuelve `False` y no modifica nada); si no, escribe `(version, valor)` en las **primeras** `W` réplicas de `alcanzables` y devuelve `True`.
2. `leer(cluster, clave, R, alcanzables)`: si hay menos de `R` alcanzables devuelve `(False, None)`; si no, consulta las **primeras** `R` de `alcanzables` y devuelve `(True, valor)` con el valor de **mayor versión** entre las que tienen la clave (`(True, None)` si ninguna la tiene).
3. `quorum_garantiza(N, R, W)`: por **fuerza bruta** (`itertools.combinations`), `True` si **todo** conjunto de escritura y **todo** conjunto de lectura se intersecan. Cuenta en `n_pares_ok` cuántos pares $(R, W)$ con $1 \le R, W \le 5$ lo garantizan y compáralo con $R + W > N$.
4. Escenario **CP** ($R = W = 3$) y **AP** ($R = W = 1$): ambos clusters parten de `("saldo", 100)` en versión 1 en las 5 réplicas. Intenta escribir `150` (versión 2) en el lado A y `999` (versión 3) en el lado B, guarda los resultados y lee según se pide en la celda.

**Criterios de aceptación:** `escribir` y `leer` pasan los casos de prueba; `quorum_garantiza` coincide con $R + W > N$; CP **rechaza** la escritura minoritaria; AP **acepta ambas**, ofrece una lectura **obsoleta** y, al sanar la red, una de las dos escrituras confirmadas **se pierde**.

**Puntaje sugerido:** 15 puntos.

In [ ]:
LADO_A, LADO_B, TODAS = [0, 1, 2], [3, 4], [0, 1, 2, 3, 4]


def nuevo_cluster(n):
    return [dict() for _ in range(n)]       # (dado)


def escribir(cluster, clave, valor, version, W, alcanzables):
    # TODO: rechaza (False) si len(alcanzables) < W; si no, escribe (version, valor) en las primeras W y devuelve True
    return None


def leer(cluster, clave, R, alcanzables):
    # TODO: (False, None) si len(alcanzables) < R; si no (True, valor de mayor versión entre las R primeras)
    return None


def quorum_garantiza(N, R, W):
    # TODO: fuerza bruta con itertools.combinations
    return None


n_pares_ok = None            # pares (R, W) con 1 <= R, W <= 5 que garantizan intersección

# Escenario CP: R = W = 3
cluster_cp = nuevo_cluster(5)
# TODO: escribe el estado inicial ("saldo", 100, versión 1) con W = 5 en TODAS
cp_a = cp_b = None           # escribir 150 (v2) en LADO_A y 999 (v3) en LADO_B, con W = 3
cp_lee_a = cp_lee_b = None   # leer "saldo" con R = 3 en LADO_A y en LADO_B

# Escenario AP: R = W = 1
cluster_ap = nuevo_cluster(5)
# TODO: escribe el estado inicial ("saldo", 100, versión 1) con W = 5 en TODAS
ap_a = ap_b = None           # escribir 150 (v2) en LADO_A y 999 (v3) en LADO_B, con W = 1
lectura_obsoleta = None      # leer "saldo" con R = 1 desde las réplicas [1, 2]
final_ap = None              # tras sanar la red: leer "saldo" con R = 5 en TODAS
escritura_perdida = None     # True si ap_a y ap_b se confirmaron pero el valor 150 ya no se lee

In [ ]:
# Autoverificación del Reto 1 (no modificar)
_c = nuevo_cluster(3)
assert escribir(_c, "k", "v1", 1, 2, [0, 1, 2]) is True and "k" in _c[0] and "k" in _c[1] and "k" not in _c[2], "escribir debe poner (versión, valor) solo en las PRIMERAS W réplicas de 'alcanzables' y devolver True."
assert escribir(_c, "k", "v2", 2, 3, [0, 1]) is False and leer(_c, "k", 2, [0, 1]) == (True, "v1"), "Con menos de W réplicas alcanzables, escribir devuelve False y NO modifica nada."
assert leer(_c, "k", 1, [2]) == (True, None), "leer con R = 1 desde una réplica que no tiene la clave devuelve (True, None)."
assert leer(_c, "k", 3, [0, 1]) == (False, None), "Con menos de R réplicas alcanzables, leer devuelve (False, None)."
_c[2]["k"] = (5, "v5")
assert leer(_c, "k", 3, [0, 1, 2]) == (True, "v5"), "leer debe devolver el valor de MAYOR versión entre las R réplicas consultadas."
assert all(quorum_garantiza(N, R, W) == (R + W > N) for N in range(1, 7) for R in range(1, N + 1) for W in range(1, N + 1)), "quorum_garantiza debe coincidir con R + W > N para todos los casos (comprueba que todo conjunto de lectura corta a todo conjunto de escritura)."
assert n_pares_ok == 15, "Con N = 5 hay 15 pares (R, W) con R + W > 5."
assert cp_a is True and cp_b is False, "CP: el lado con mayoría (3 de 5) escribe; el lado minoritario (2 de 5) debe RECHAZAR."
assert cp_lee_a == (True, 150) and cp_lee_b == (False, None), "CP: el lado A lee 150; el lado B no alcanza el quórum de lectura y no sirve datos."
assert ap_a is True and ap_b is True, "AP: con W = 1 ambos lados aceptan la escritura."
assert lectura_obsoleta == (True, 100), "AP: leer con R = 1 desde réplicas que no recibieron la escritura devuelve el valor viejo (100)."
assert final_ap == (True, 999), "Al sanar la red, leer con R = 5 devuelve el valor de mayor versión (999)."
assert escritura_perdida is True, "AP: ambas escrituras se confirmaron pero al reconciliar por versión el 150 se pierde (escritura_perdida = True)."
print(f"✅ Reto 1 superado: {n_pares_ok}/25 pares (R, W) con quórum | CP rechaza la minoría | AP: lectura obsoleta {lectura_obsoleta[1]} y escritura perdida (valor final {final_ap[1]})")

---
## Reto 2: MapReduce desde cero, *combiner* y sesgo de claves 🗺️

**Contexto:** quieres, por municipio, la **suma**, el **conteo** y el **máximo** de kWh de 24 000 lecturas; de ahí sale el promedio. La capital concentra el 55 % de las lecturas, así que la clave `"Tunja"` es una **clave caliente**.

**Enunciado**
1. `mapear(trozo)`: por cada `(municipio, kwh)` emite `(municipio, (kwh, 1, kwh))` (suma, conteo, máximo).
2. `combinar(pares)`: agrega **localmente** por clave y devuelve una lista de `(clave, (suma, conteo, máximo))`.
3. `reducir(valores)`: combina una lista de tripletas `(suma, conteo, máximo)` en una sola.
4. `particion(clave, n_red)`: reductor destino $= \text{crc32}(\text{str(clave)}) \bmod n_{red}$ (con `zlib.crc32`, estable entre ejecuciones).
5. `ejecutar_mr(registros, n_map, n_red, usar_combiner, sal)` → `(resultado, n_barajados, cargas)`: divide con `dividir`, aplica `mapear`, **sala** la clave `k` como `(k, j % sal)` con `j` la posición del par dentro de su trozo, aplica `combinar` si corresponde, reparte con `particion` (*shuffle*), reduce en cada reductor y **fusiona las sales** por municipio. `n_barajados` es el total de pares enviados al *shuffle* y `cargas[r]` los pares que recibe el reductor `r`.
6. Con `lecturas`: `ejecutar_mr(lecturas, 4, 4, True, 1)` → `res_con, n_con, _`; sin *combiner* → `res_sin, n_sin, cargas_sin`; sin *combiner* y `sal=16` → `res_sal, _, cargas_sal`. Calcula `desb_sin` y `desb_sal` (máximo de las cargas / promedio) y `promedios` (municipio → promedio de kWh) desde `res_con`.

**Criterios de aceptación:** el resultado coincide con `pandas.groupby`; el *combiner* reduce los pares barajados más de 100 veces; sin sal el reductor más cargado recibe más de 2 veces el promedio; con sal el desbalance baja de 1.3 **sin cambiar el resultado**.

**Puntaje sugerido:** 25 puntos.

In [ ]:
lecturas = generar_lecturas()


def mapear(trozo):
    # TODO: [(municipio, (kwh, 1, kwh)), ...]
    return None


def combinar(pares):
    # TODO: agrega por clave -> lista de (clave, (suma, conteo, máximo))
    return None


def reducir(valores):
    # TODO: combina tripletas (suma, conteo, máximo) en una sola
    return None


def particion(clave, n_red):
    # TODO: crc32 de str(clave) módulo n_red
    return None


def ejecutar_mr(registros, n_map=4, n_red=4, usar_combiner=True, sal=1):
    # TODO: map -> (sal) -> combiner opcional -> shuffle -> reduce -> fusionar sales
    # devuelve (resultado {municipio: (suma, conteo, máximo)}, n_barajados, cargas)
    return None


# TODO: completa
res_con, n_con, _ = None, None, None              # con combiner, sal = 1
res_sin, n_sin, cargas_sin = None, None, None     # sin combiner, sal = 1
res_sal, _, cargas_sal = None, None, None         # sin combiner, sal = 16
desb_sin = desb_sal = None                        # max(cargas) / promedio(cargas)
promedios = None                                  # {municipio: promedio de kWh} desde res_con

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_toy = [("a", 1.0), ("b", 2.0), ("a", 3.0)]
assert mapear(_toy) == [("a", (1.0, 1, 1.0)), ("b", (2.0, 1, 2.0)), ("a", (3.0, 1, 3.0))], "mapear debe emitir (municipio, (kwh, 1, kwh)) por cada lectura, en el mismo orden."
assert dict(combinar(mapear(_toy))) == {"a": (4.0, 2, 3.0), "b": (2.0, 1, 2.0)}, "combinar debe sumar sumas y conteos y quedarse con el máximo, por clave."
assert reducir([(1.0, 1, 1.0), (3.0, 1, 3.0)]) == (4.0, 2, 3.0), "reducir devuelve (suma de sumas, suma de conteos, máximo de máximos)."
assert particion("Tunja", 4) == zlib.crc32(b"Tunja") % 4 and particion(("Tunja", 3), 4) == zlib.crc32(str(("Tunja", 3)).encode()) % 4, "particion = zlib.crc32(str(clave).encode()) % n_red."
_df = pd.DataFrame(lecturas, columns=["m", "kwh"]).groupby("m")["kwh"].agg(["sum", "count", "max"])
for _nom, _res in [("con combiner", res_con), ("sin combiner", res_sin), ("con sal", res_sal)]:
    assert isinstance(_res, dict) and set(_res) == set(_df.index), f"{_nom}: el resultado debe tener una entrada por municipio (fusiona las sales al final)."
    assert all(abs(_res[m][0] - r["sum"]) < 1e-6 * r["sum"] and _res[m][1] == r["count"] and abs(_res[m][2] - r["max"]) < 1e-9 for m, r in _df.iterrows()), f"{_nom}: suma, conteo y máximo deben coincidir con pandas.groupby."
assert n_sin == len(lecturas) == sum(cargas_sin), "Sin combiner cada lectura viaja al shuffle: n_barajados = número de lecturas = suma de las cargas."
assert n_con <= 4 * len(MUNICIPIOS) and n_sin / n_con > 100, "Con combiner cada mapper envía a lo sumo un par por municipio (4 × 12 = 48): el ahorro debe superar 100 veces."
assert len(cargas_sin) == 4 and sum(cargas_sal) == len(lecturas), "cargas debe tener un valor por reductor y sumar los pares barajados."
assert abs(desb_sin - max(cargas_sin) / np.mean(cargas_sin)) < 1e-9 and desb_sin > 2, "Sin sal, la clave caliente 'Tunja' cae en un solo reductor: el desbalance debe superar 2."
assert abs(desb_sal - max(cargas_sal) / np.mean(cargas_sal)) < 1e-9 and desb_sal < 1.3, "Con sal = 16 la clave caliente se reparte entre los 4 reductores: el desbalance debe bajar de 1.3."
assert promedios is not None and all(abs(promedios[m] - r["sum"] / r["count"]) < 1e-9 for m, r in _df.iterrows()), "promedios[m] = suma / conteo (no un promedio de promedios)."
print(f"✅ Reto 2 superado: pares barajados {n_sin:,} → {n_con} con combiner | desbalance {desb_sin:.2f} → {desb_sal:.2f} con sal | Tunja promedia {promedios['Tunja']:.1f} kWh")

---
## Reto 3: Un mini-HDFS que sobrevive a la caída de un *rack* 🗄️

**Contexto:** guardarás un archivo de `TAM_ARCHIVO_MB = 1000` MB en un clúster de 6 DataNodes repartidos en dos *racks* (`RACKS`), con bloques de `BLOQUE_MB = 128` MB y factor de replicación `FACTOR = 3`. El NameNode es un diccionario `namenode = {id_bloque: [nodos con réplica]}`.

**Enunciado**
1. `dividir_bloques(tam_mb, bloque_mb)`: lista con el tamaño de cada bloque (el último puede ser menor).
2. `colocar(i, factor)`: réplicas del bloque `i`: la **primera** en `NODOS[i % 6]`; la **segunda** en un nodo del **otro *rack*** (el de posición `i % 3` entre los nodos de ese *rack*); la **tercera** en el **mismo *rack* que la segunda**, en otro nodo (el siguiente de forma circular). Devuelve solo las primeras `factor`.
3. `caer(nn, caidos)`: copia del NameNode sin las réplicas de los nodos caídos; `bloques_perdidos(nn)`: bloques sin ninguna réplica.
4. `re_replicar(nn, vivos, factor)`: devuelve una copia en la que cada bloque **no perdido** recupera réplicas hasta tener `min(factor, len(vivos))`, siempre en nodos vivos que no lo tengan, eligiendo el de menor carga (bloques almacenados; empate: nombre menor).
5. `frac_perdida(nn, k)`: fracción de las combinaciones de `k` nodos caídos simultáneamente que pierden **al menos un bloque**.
6. Con ellas: `bloques`, `namenode`; falla de `dn1` (`nn_nodo`, `nn_nodo_rep`); falla del *rack* A completo (`nn_rack`, `perdidos_rack`, `nn_rack_rep`); `frac3_k2` y `frac3_k3` (factor 3) y `frac2_k2` (factor 2); y `ratio_metadatos`: bloques que tendría 1 archivo de 10 000 MB dividido entre los que tendrían 10 000 archivos de 1 MB.

**Criterios de aceptación:** la colocación cumple la regla; ningún bloque se pierde por la caída de un nodo ni de un *rack*; tras re-replicar todo bloque vuelve a su factor; con factor 3 dos nodos caídos nunca pierden datos pero con factor 2 sí; y los archivos pequeños multiplican los metadatos más de 100 veces.

**Puntaje sugerido:** 25 puntos.

In [ ]:
RACKS = {"dn1": "rackA", "dn2": "rackA", "dn3": "rackA", "dn4": "rackB", "dn5": "rackB", "dn6": "rackB"}
NODOS = list(RACKS)
TAM_ARCHIVO_MB, BLOQUE_MB, FACTOR = 1000, 128, 3


def dividir_bloques(tam_mb, bloque_mb):
    # TODO: tamaños de los bloques (el último puede ser menor)
    return None


def colocar(i, factor=FACTOR):
    # TODO: [primera réplica, otro rack, mismo rack que la segunda][:factor]
    return None


def caer(nn, caidos):
    # TODO: copia sin las réplicas de los nodos caídos
    return None


def bloques_perdidos(nn):
    # TODO: ids de bloques sin réplicas
    return None


def re_replicar(nn, vivos, factor):
    # TODO: devuelve una copia con las réplicas faltantes repuestas en nodos vivos de menor carga
    return None


def frac_perdida(nn, k):
    # TODO: fracción de combinaciones de k nodos caídos que pierden al menos un bloque
    return None


# TODO: completa
bloques = namenode = None
nn_nodo = nn_nodo_rep = None          # falla de "dn1" y su re-replicación (vivos = los otros 5)
caidos_rack = ["dn1", "dn2", "dn3"]
nn_rack = perdidos_rack = nn_rack_rep = None   # falla del rack A completo; re-replicar sobre dn4-dn6
frac3_k2 = frac3_k3 = frac2_k2 = None  # fracciones de pérdida (factor 3 con k=2 y k=3; factor 2 con k=2)
ratio_metadatos = None                 # bloques (10 000 archivos de 1 MB) / bloques (1 archivo de 10 000 MB)

In [ ]:
# Autoverificación del Reto 3 (no modificar)
assert dividir_bloques(1000, 128) == [128] * 7 + [104] and dividir_bloques(128, 128) == [128] and dividir_bloques(1, 128) == [1], "dividir_bloques(1000, 128) debe dar 7 bloques de 128 y uno de 104; si el archivo es múltiplo exacto no hay bloque extra."
assert bloques == [128] * 7 + [104] and sum(bloques) == TAM_ARCHIVO_MB, "bloques = dividir_bloques(TAM_ARCHIVO_MB, BLOQUE_MB) (8 bloques que suman 1000 MB)."
for _i in range(12):
    _r = colocar(_i, 3)
    assert _r is not None and len(_r) == 3 and len(set(_r)) == 3, f"colocar({_i}) debe devolver 3 nodos distintos."
    assert _r[0] == NODOS[_i % 6] and RACKS[_r[1]] != RACKS[_r[0]] and RACKS[_r[2]] == RACKS[_r[1]], f"colocar({_i}): la 1.ª réplica en NODOS[i % 6], la 2.ª en el otro rack y la 3.ª en el rack de la 2.ª."
assert colocar(0, 1) == [NODOS[0]] and len(colocar(0, 2)) == 2 and RACKS[colocar(0, 2)[0]] != RACKS[colocar(0, 2)[1]], "Con factor 1 o 2 se devuelven solo las primeras réplicas (factor 2: dos racks distintos)."
assert namenode == {i: colocar(i) for i in range(8)}, "namenode = {i: colocar(i) para cada bloque}."
assert caer({0: ["a", "b"], 1: ["b"]}, ["b"]) == {0: ["a"], 1: []} and bloques_perdidos({0: ["a"], 1: []}) == [1], "caer quita las réplicas de los nodos caídos (sin modificar el original); bloques_perdidos lista los bloques con lista vacía."
assert all("dn1" not in v for v in nn_nodo.values()) and min(len(v) for v in nn_nodo.values()) == 2 and "dn1" in namenode[0], "Tras caer dn1, ningún bloque lo referencia y los demás conservan 2 réplicas; el namenode original no debe modificarse."
assert not bloques_perdidos(nn_nodo), "Con factor 3, la caída de un nodo no pierde ningún bloque."
_vivos = [n for n in NODOS if n != "dn1"]
assert all(len(v) == 3 and len(set(v)) == 3 and set(v) <= set(_vivos) for v in nn_nodo_rep.values()), "re_replicar: cada bloque debe volver a 3 réplicas distintas, todas en nodos vivos."
assert all(set(nn_nodo[b]) <= set(nn_nodo_rep[b]) for b in nn_nodo), "re_replicar debe conservar las réplicas sobrevivientes (solo añade)."
assert perdidos_rack == [] and all(1 <= len(v) <= 2 for v in nn_rack.values()), "Con réplicas en dos racks, caer un rack completo deja cada bloque con 1 o 2 réplicas y ninguno perdido."
assert all(sorted(v) == ["dn4", "dn5", "dn6"] for v in nn_rack_rep.values()), "Con solo 3 nodos vivos, re_replicar debe dejar cada bloque en dn4, dn5 y dn6 (min(factor, nº de vivos) = 3)."
assert re_replicar({0: ["dn4"], 1: []}, ["dn4", "dn5"], 3) == {0: ["dn4", "dn5"], 1: []}, "Un bloque perdido (sin réplicas) no se puede reconstruir: debe quedar vacío; un bloque con 1 réplica y 2 nodos vivos queda con 2."
assert frac3_k2 == 0.0 and frac3_k3 > 0 and frac2_k2 > 0, "Con factor 3, 2 nodos caídos no pierden datos (0.0); con 3 caídos o con factor 2 sí puede haber pérdida (> 0)."
assert abs(frac3_k3 - frac_perdida({i: colocar(i) for i in range(8)}, 3)) < 1e-12 and 0 < frac2_k2 < 1, "Las fracciones deben salir de frac_perdida sobre la colocación correspondiente."
assert abs(ratio_metadatos - 10_000 / 79) < 1e-9 and ratio_metadatos > 100, "10 000 archivos de 1 MB necesitan 10 000 bloques; 1 archivo de 10 000 MB solo 79."
print(f"✅ Reto 3 superado: sin pérdidas al caer un nodo o un rack | fracción con pérdida: factor 3 (k=2) {frac3_k2:.2f}, factor 3 (k=3) {frac3_k3:.2f}, factor 2 (k=2) {frac2_k2:.2f} | archivos pequeños: {ratio_metadatos:.0f}× más metadatos")

---
## Reto 4: Un mini-HBase y el punto caliente 🔥

**Contexto:** HBase guarda filas **ordenadas por clave**, con varias **versiones** por celda y divide la tabla en **regiones** contiguas. Si las claves son monótonas (ids crecientes), todas las escrituras nuevas caen en la **última región**: un **punto caliente**. Una solución es **salar** la clave, a costa de escanear varias veces.

**Enunciado**
1. Clase `MiniHBase(max_versiones=3)` con `put(fila, col, valor, ts)` (mantiene `claves` ordenadas con `bisect`; guarda a lo sumo `max_versiones` versiones por celda, descartando las más antiguas; mismo `ts` = reemplazo), `versiones(fila, col)` (lista `(ts, valor)` de la más reciente a la más antigua), `get(fila, col, ts_max=None)` (valor más reciente con `ts <= ts_max`; `None` si no hay), `scan(inicio, fin)` (filas en `[inicio, fin)`, ordenadas, con `bisect`) y `scan_prefijo(prefijo)`.
2. `region_de(clave, inicios)`: índice de la región que contiene la clave (`inicios` ordenado; la región `r` va de `inicios[r]` hasta antes de `inicios[r+1]`).
3. Una tabla ya dividida en 4 regiones recibe 2 000 escrituras nuevas con ids `1000..2999`. Con clave monótona `f"{id:08d}"` calcula `carga_mono` (escrituras por región) y con clave salada `clave_sal(id)` $= $ `f"{crc32(str(id)) % 4:02d}-{id:08d}"` calcula `carga_sal`. Obtén `desb_mono` y `desb_sal` (máximo / promedio).
4. Carga las 2 000 filas saladas en `tabla_sal` (celda `"cf:v"`, valor = id, `ts=1`) y escribe `scan_con_sal(tabla, id_ini, id_fin, n_sal=4)` → `(ids, n_scans)`: ids en `[id_ini, id_fin)` en orden ascendente, haciendo **un escaneo por sal**.

**Criterios de aceptación:** `MiniHBase` pasa los casos de prueba; la clave monótona manda el 100 % de la carga a una región (desbalance 4.0); la salada queda por debajo de 1.3; y el escaneo con sal devuelve exactamente el mismo rango, con 4 escaneos.

**Puntaje sugerido:** 20 puntos.

In [ ]:
class MiniHBase:
    def __init__(self, max_versiones=3):
        self.max_versiones = max_versiones
        self.claves = []     # filas, siempre ordenadas
        self.celdas = {}     # (fila, columna) -> lista de (ts, valor)

    def put(self, fila, col, valor, ts):
        # TODO: inserta la fila en orden si es nueva; guarda (ts, valor); conserva a lo sumo max_versiones
        pass

    def versiones(self, fila, col):
        # TODO: [(ts, valor), ...] de la más reciente a la más antigua ([] si no existe)
        return None

    def get(self, fila, col, ts_max=None):
        # TODO: valor más reciente con ts <= ts_max (o el más reciente si ts_max es None); None si no hay
        return None

    def scan(self, inicio, fin):
        # TODO: filas en [inicio, fin), ordenadas, usando bisect
        return None

    def scan_prefijo(self, prefijo):
        # TODO: filas que empiezan con `prefijo`
        return None


def region_de(clave, inicios):
    # TODO: índice de la región que contiene la clave
    return None


inicios = ["", "00000250", "00000500", "00000750"]   # tabla dividida en 4 regiones
ids_nuevos = range(1000, 3000)

# TODO: completa
carga_mono = None     # [escrituras en región 0, 1, 2, 3] con clave f"{id:08d}"


def clave_sal(i, n_sal=4):
    # TODO: f"{crc32(str(i)) % n_sal:02d}-{i:08d}"
    return None


inicios_sal = ["", "01-", "02-", "03-"]
carga_sal = None      # escrituras por región con clave_sal
desb_mono = desb_sal = None

tabla_sal = None      # MiniHBase con las 2 000 filas saladas (celda "cf:v" = id, ts = 1)


def scan_con_sal(tabla, id_ini, id_fin, n_sal=4):
    # TODO: un escaneo por sal; devuelve (ids ordenados en [id_ini, id_fin), n_scans)
    return None


ids_rango, n_scans = None, None   # scan_con_sal(tabla_sal, 1500, 1600)

In [ ]:
# Autoverificación del Reto 4 (no modificar)
_t = MiniHBase(max_versiones=2)
for _f, _ts, _v in [("b", 1, "b1"), ("a", 1, "a1"), ("c", 1, "c1"), ("a", 2, "a2"), ("a", 3, "a3"), ("ab", 1, "ab1")]:
    _t.put(_f, "cf:x", _v, _ts)
assert _t.claves == ["a", "ab", "b", "c"], "put debe mantener 'claves' ordenadas y sin repetir filas (usa bisect.insort solo si la fila es nueva)."
assert _t.versiones("a", "cf:x") == [(3, "a3"), (2, "a2")], "Con max_versiones = 2 solo se conservan las 2 versiones más recientes, de la más nueva a la más vieja."
assert _t.get("a", "cf:x") == "a3" and _t.get("a", "cf:x", ts_max=2) == "a2" and _t.get("a", "cf:x", ts_max=1) is None, "get devuelve el valor más reciente con ts <= ts_max; None si ninguna versión cumple."
assert _t.get("zz", "cf:x") is None and _t.versiones("zz", "cf:x") == [], "Una celda inexistente da None / lista vacía."
_t.put("b", "cf:x", "b1-nuevo", 1)
assert _t.versiones("b", "cf:x") == [(1, "b1-nuevo")], "Escribir con el mismo ts reemplaza la versión, no la duplica."
assert _t.scan("a", "b") == ["a", "ab"] and _t.scan("ab", "zz") == ["ab", "b", "c"] and _t.scan("x", "z") == [], "scan(inicio, fin) devuelve las filas en [inicio, fin), ordenadas."
assert _t.scan_prefijo("a") == ["a", "ab"] and _t.scan_prefijo("c") == ["c"] and _t.scan_prefijo("q") == [], "scan_prefijo devuelve las filas que empiezan con el prefijo."
assert [region_de(k, inicios) for k in ["00000000", "00000249", "00000250", "00000999", "00009999"]] == [0, 0, 1, 3, 3], "region_de: la región r contiene las claves desde inicios[r] hasta antes de inicios[r+1] (bisect_right - 1)."
assert carga_mono == [0, 0, 0, 2000] and abs(desb_mono - 4.0) < 1e-9, "Con claves monótonas todas las escrituras nuevas caen en la última región: [0, 0, 0, 2000] y desbalance 4.0."
assert clave_sal(1234) == f"{zlib.crc32(b'1234') % 4:02d}-00001234", "clave_sal(i) = f'{crc32(str(i)) % 4:02d}-{i:08d}'."
assert sum(carga_sal) == 2000 and len(carga_sal) == 4 and abs(desb_sal - max(carga_sal) / 500) < 1e-9 and desb_sal < 1.3, "Con sal, la carga debe repartirse entre las 4 regiones (desbalance < 1.3)."
assert tabla_sal is not None and len(tabla_sal.claves) == 2000 and tabla_sal.claves == sorted(tabla_sal.claves), "tabla_sal debe contener las 2 000 filas saladas, ordenadas."
assert ids_rango == list(range(1500, 1600)) and n_scans == 4, "scan_con_sal debe devolver los ids 1500..1599 en orden con 4 escaneos (uno por sal)."
assert scan_con_sal(tabla_sal, 2990, 3010)[0] == list(range(2990, 3000)), "Un rango que se sale de la tabla solo devuelve los ids existentes."
print(f"✅ Reto 4 superado: carga monótona {carga_mono} (desbalance {desb_mono:.1f}) → con sal {carga_sal} (desbalance {desb_sal:.2f}) | rango salado en {n_scans} escaneos")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Qué pasó con el lado minoritario en CP y en AP? ¿Qué sacrifica cada política (consistencia o disponibilidad) y qué papel jugó $R + W > N$ en tu `n_pares_ok`?
- ¿Cuánto bajó el número de pares barajados con el *combiner* y por qué el *combiner* **no** arregla el sesgo cuando la operación no es combinable? ¿Qué costo tiene la sal?
- ¿Por qué un *rack* completo no perdió datos con factor 3 y qué habría pasado con todas las réplicas en el mismo *rack*? ¿Qué te dicen `frac3_k2`, `frac3_k3` y `frac2_k2` sobre elegir el factor?
- ¿Por qué los archivos pequeños son un problema para el NameNode (usa `ratio_metadatos`)?
- En HBase, ¿por qué las claves monótonas crean un punto caliente y qué pagas con las claves saladas al leer rangos?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: lectura y escritura con quórum, fuerza bruta de $R + W > N$ y comparación CP frente a AP | 15 |
| Reto 2: MapReduce completo con *combiner*, partición por *hash* y sal contra el sesgo | 25 |
| Reto 3: mini-HDFS con *racks*, fallas, re-replicación y archivos pequeños | 25 |
| Reto 4: mini-HBase (versiones, rangos, prefijos) y punto caliente con sal | 20 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Verifiqué el resultado de MapReduce contra `pandas.groupby` y la fusión de sales.
- [ ] Probé la caída de un nodo y de un *rack* sin perder bloques.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Big Data</i>
  </p>
</div>